# Kochi-LLM v1.0 — publish checkpoints to Hugging Face

Converts the most-trained checkpoint of **each attached Kaggle run** to bf16 safetensors and pushes it to
`<your-hf-username>/KochiLLM-v1.0` as its own git revision, tagged `run-NN`.

The final weights also land on `main`. `main` is only overwritten when this run has a **higher step count**
than what is already published, so the notebook is **order-independent** and safe to re-run.

## Why you may need several sessions

Each checkpoint is 4.1 GB, so attaching all nine at once (~37 GB) will likely exceed Kaggle's notebook
input limit. Attach as many runs as the limit allows, run this notebook, then attach the next batch and
run it again. Already-published revisions are simply overwritten with identical content.

## Setup

- Attach one or more Kaggle notebook outputs that contain `best_model.pth` / `latest_backup_model.pth`
- **Internet ON**, GPU not required (all work is CPU + network)
- Add an `HF_TOKEN` Kaggle secret with write access to your target repo


In [ ]:
%%bash
set -e
pip install -q -U huggingface_hub safetensors

DIRS=$(find /kaggle/input -name 'best_model.pth' -o -name 'latest_backup_model.pth' | xargs -r -n1 dirname | sort -u)
echo "$DIRS"
N=$(printf '%s' "$DIRS" | grep -c . || true)
echo "found $N run dir(s)"
[ "$N" -eq 0 ] && { echo 'NO RUNS ATTACHED'; ls -R /kaggle/input | head -40; exit 1; }

PY_SRC=''
for d in $DIRS /kaggle/input/datasets/dadhichisarker/kochi-llm/model; do
  if [ -f "$d/architecture.py" ]; then PY_SRC=$d; break; fi
done
[ -z "$PY_SRC" ] && { echo 'NO PY SOURCE'; exit 1; }
cp "$PY_SRC"/*.py /kaggle/working/
find /kaggle/input -name 'custom_bpe_tokenizer.pkl' -exec cp {} /kaggle/working/ \; 2>/dev/null || true
ls -lh /kaggle/working/*.py /kaggle/working/*.pkl 2>/dev/null || true


## Before uploading

Check the table below: each row is one attached run, and `chosen` is the checkpoint with the
**highest step count** — not necessarily `best_model.pth`, because several early runs carry a stale
`best_model.pth` from Phase 1 while their real weights sit in `latest_backup_model.pth`.


In [ ]:
import glob, os, re
import torch

WEIGHTS = ('best_model.pth', 'latest_backup_model.pth')

dirs = sorted({os.path.dirname(p)
               for p in glob.glob('/kaggle/input/**/*.pth', recursive=True)
               if os.path.basename(p) in WEIGHTS})


def tag_for(slug, idx):
    nums = re.findall(r'\d+', slug)
    return 'run-%02d' % (int(nums[-1]) if nums else idx)


plan = []
for i, d in enumerate(dirs, 1):
    cands = []
    for n in WEIGHTS:
        fp = os.path.join(d, n)
        if os.path.exists(fp):
            ck = torch.load(fp, map_location='cpu', weights_only=False, mmap=True)
            cands.append((int(ck.get('iter', -1)), fp, ck))
    if not cands:
        continue
    it, fp, ck = max(cands, key=lambda t: t[0])
    slug = os.path.basename(d.rstrip('/'))
    plan.append({'dir': d, 'slug': slug, 'branch': tag_for(slug, i),
                 'iter': it, 'path': fp, 'ckpt': ck})

print('%-36s %-8s %7s %8s  %s' % ('source run', 'branch', 'iter', 'val', 'chosen file'))
for e in sorted(plan, key=lambda e: e['iter']):
    v = e['ckpt'].get('val_loss')
    v = float(v) if v is not None else float('nan')
    print('%-36s %-8s %7d %8.4f  %s' % (e['slug'][:36], e['branch'], e['iter'], v,
                                        os.path.basename(e['path'])))
print('\n%d run(s) will be pushed.' % len(plan))


In [ ]:
import os
from huggingface_hub import HfApi

TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no Kaggle secret:', e)
if not TOKEN:
    TOKEN = os.environ.get('HF_TOKEN')
if not TOKEN:
    raise SystemExit('Add an HF_TOKEN Kaggle secret (write access) and re-run.')

api = HfApi(token=TOKEN)
USER = api.whoami()['name']
REPO = USER + '/KochiLLM-v1.0'
api.create_repo(REPO, repo_type='model', exist_ok=True)
print('target repo:', REPO)

MODEL_CARD = '''---
library_name: pytorch
tags:
  - moe
  - mixture-of-experts
  - gqa
  - from-scratch
  - custom-architecture
language:
  - en
license: mit
---

# KochiLLM v1.0

A 1.02B-parameter Mixture-of-Experts language model **trained entirely from scratch** on 2x Tesla T4
(~52 GPU-hours), including its own BPE tokenizer and data pipeline.

| | |
|---|---|
| Total parameters | 1,023M (~634M active per token) |
| Layers | 12, dim 1536 |
| Attention | 24 query / 8 KV heads (GQA), head_dim 64, RoPE, QK-Norm |
| Experts | 4 routed (top-2) + 1 shared, SwiGLU (hidden = 2 x dim) |
| Attention window | alternating: full (even layers), 512-token sliding (odd layers) |
| Soft-capping | attention logits 50, final logits 30 |
| Multi-token prediction | 2 heads; extra head discarded at inference |
| Tokenizer | custom BPE, 32,000 vocab, regex pre-tokenization |
| Context length | 1024 tokens |
| Training tokens | 180,224,000 (0.9996 epochs of FineWeb-Edu + Cosmopedia-v2, 4:1) |
| Final validation loss | **3.96** |

## Loading

This is **not** an `AutoModel` release: the tokenizer is a custom pickle and the architecture is
custom code. Load it directly:

```python
import json, torch
from safetensors.torch import load_file
from architecture import MyCustomLLM
from tokenizer import MyCustomTokenizer

config = json.load(open('config.json'))
tok = MyCustomTokenizer(vocab_size=config['vocab_size'])
tok.load('custom_bpe_tokenizer.pkl')

model = MyCustomLLM(**config)          # re-ties embed.weight <-> output.weight
sd = load_file('model.safetensors')
sd['output.weight'] = sd['embed.weight']  # dropped at save time; tied in __init__
missing, unexpected = model.load_state_dict(sd, strict=False)
model.eval().cuda().to(torch.bfloat16)

print(tok.decode(tok.encode('The capital of France is')))
```

`generate.py` provides KV-cache sampling (`generate(model, tok, prompt, ...)`).

## Revisions

Every training run is preserved as its own revision, tagged `run-NN`, with `metadata.json` recording
its step count, validation loss, and source run. `main` holds the most-trained checkpoint
(highest step count), which is `latest_backup_model.pth` at step 5499, val 3.96.

## Limitations

Read these before drawing conclusions from the output:

- **No factual knowledge.** At 0.176 tokens/param the model learned fluency and style, not facts.
  Compute-optimal for 1B parameters is ~20 tokens/param (~20B tokens); this run is deliberately
  data-limited. Generated names and dates are plausible and wrong.
- **Not instruction-tuned.** Trained only on web pages and textbook prose, so it continues documents
  rather than answering questions.
- **Short context.** 1024 tokens.
- **Weights are bf16.** Cast down from the fp32 training checkpoint to halve size; use it for
  inference, not for resuming training.
- **Optimizer state was not checkpointed** across Kaggle session boundaries, so AdamW moments reset
  roughly four times during training.
'''

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(MODEL_CARD)

CODE_FILES = ['architecture.py', 'tokenizer.py', 'data_loader.py',
              'generate.py', 'prepare_data.py', 'train_custom.py',
              'custom_bpe_tokenizer.pkl']
uploaded = []
for f in CODE_FILES:
    if os.path.exists(f):
        api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO,
                        commit_message='Add ' + f)
        uploaded.append(f)
api.upload_file(path_or_fileobj='README.md', path_in_repo='README.md', repo_id=REPO,
                commit_message='Add model card')
print('uploaded code:', uploaded)
print('uploaded model card')


## Notes

- **Tied weights:** the input embedding and output projection share storage, which `safetensors`
  refuses to serialise twice. `output.weight` is dropped and noted in `metadata.json`; rebuild it with
  `model.embed.weight = model.output.weight` (already done inside `MyCustomLLM.__init__`).
- **bf16:** uploaded weights are cast down from fp32, halving size to ~2.05 GB. They are for
  inference, not for resuming training.
- **This is not an `AutoModel` release.** The tokenizer is a custom pickle and the architecture is
  custom, so load it the way the model card describes:
  `MyCustomLLM(**config)` then `load_state_dict` from the safetensors file.

In [ ]:
import gc, json, os
import torch
from safetensors.torch import save_file
from huggingface_hub import CommitOperationAdd, hf_hub_download


def build_ops(ckpt, entry):
    sd = ckpt['model_state_dict']
    seen, kept, dropped = {}, {}, []
    for k, v in sd.items():
        ptr = v.data_ptr()
        if ptr in seen:
            dropped.append(k)
            continue
        seen[ptr] = k
        kept[k] = v.to(torch.bfloat16).contiguous().cpu()
    save_file(kept, 'model.safetensors', metadata={'format': 'pt'})

    json.dump(ckpt.get('config', {}), open('config.json', 'w'), indent=2)
    meta = {
        'iter': int(ckpt.get('iter', -1)),
        'val_loss': float(ckpt.get('val_loss', float('nan'))),
        'latest_val_loss': float(ckpt.get('latest_val_loss', float('nan'))),
        'source_run': entry['slug'],
        'dtype': 'bfloat16',
        'tied_weights_dropped': dropped,
        'note': 'embed.weight is tied to output.weight; dropped here to satisfy safetensors',
    }
    json.dump(meta, open('metadata.json', 'w'), indent=2)

    return [
        CommitOperationAdd(path_or_fileobj='model.safetensors', path_in_repo='model.safetensors'),
        CommitOperationAdd(path_or_fileobj='config.json', path_in_repo='config.json'),
        CommitOperationAdd(path_or_fileobj='metadata.json', path_in_repo='metadata.json'),
    ]


main_iter = -1
try:
    existing = hf_hub_download(REPO, 'metadata.json', revision='main')
    main_iter = json.load(open(existing)).get('iter', -1)
except Exception:
    pass
print('main is currently at iter', main_iter)
print('pushing', len(plan), 'run(s) to', REPO)

for e in sorted(plan, key=lambda e: e['iter']):
    branch = e['branch']
    ops = build_ops(e['ckpt'], e)
    val = float(e['ckpt'].get('val_loss', float('nan')))
    msg = '%s: iter %d, val %.4f (from %s)' % (branch, e['iter'], val, e['slug'])

    try:
        api.create_branch(REPO, branch=branch, exist_ok=True)
    except Exception as ex:
        print('  branch create skipped:', ex)

    api.create_commit(REPO, operations=ops, commit_message=msg, revision=branch)
    api.create_tag(REPO, tag=branch, revision=branch)
    print('pushed', branch, '| iter', e['iter'], '| val %.4f' % val)

    if e['iter'] >= main_iter:
        api.create_commit(REPO, operations=ops, commit_message=msg + ' [main]', revision='main')
        print('  main updated to iter', e['iter'])
        main_iter = e['iter']

    for f in ('model.safetensors', 'config.json', 'metadata.json'):
        if os.path.exists(f):
            os.remove(f)
    del ops
    gc.collect()

print('done')


In [ ]:
refs = api.list_repo_refs(REPO)
print('repo:', REPO)
print('branches:', [b.name for b in refs.branches])
print('tags:     ', [t.name for t in refs.tags])
print('\nfiles on main:')
for s in api.repo_info(REPO).siblings:
    print('  ', s.rfilename)
